<div style='background:#1a1a2e;color:#eee;padding:18px;border-radius:8px;margin-bottom:12px'>
<h2 style='margin:0'>🍎 EDUCATOR VERSION — Chapter 10: Observational Studies</h2>
<p style='margin:6px 0 0'>This notebook contains answer keys, teaching notes, and discussion prompts. Students should use the standard <strong>Chapter_10.ipynb</strong>.</p>
</div>


# Chapter 10: Observational Studies in Structural Engineering
## Learning From Structures We Cannot Control

**Learning Objectives:**
- Distinguish observational studies from controlled experiments
- Identify confounding variables in real structural load data
- Explain why correlation between structural measurements does not imply causation
- Connect post-collapse investigations to observational study methodology


<center>
<img src='https://upload.wikimedia.org/wikipedia/commons/8/83/I35W_bridge_collapse_TLR1.jpg' width='700' />

<em>The I-35W Mississippi River Bridge, Minneapolis, seconds after collapse on August 1, 2007. Thirteen people died; the investigation became a landmark in observational structural analysis. (NTSB — public domain.)</em>
</center>

---


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Context

**Curriculum connections:**
- *AP Statistics*: Unit 2 (Exploring Two-Variable Data — correlation, scatter plots); Unit 3 (Collecting Data — observational studies vs. experiments, confounding)
- *AP Physics*: Experimental design; controlling variables; sources of measurement error
- *Statistics / Math class*: Correlation, causation, lurking variables

**Prerequisites:** Students should understand correlation (r), scatter plots, and basic experimental design vocabulary (treatment group, control group, explanatory variable, response variable).

**Estimated time:** 50 minutes. Can split Widget 1 + reflection (25 min) and case study + Widget 2 (25 min) across two class periods.

**Pedagogical note:** This chapter deliberately uses structural engineering to make confounding *physical* rather than abstract. Students feel the intuition — *of course* traffic makes the bridge deflect! — which makes the temperature-as-confound reveal more satisfying. The I-35W collapse is emotionally anchored (many students have seen the footage online) and reinforces that these are not toy examples.
</div>


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ipywidgets','--quiet'])
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
np.random.seed(7)

# Simulate 300 days of bridge monitoring data
# True model: midspan deflection (mm) = 0.04*traffic + 0.15*temperature + noise
# Traffic (vehicles/hour) and temperature (°F) are correlated — more traffic in warm months
n = 300
temperature = np.random.normal(55, 22, n)          # °F, annual cycle approximation
traffic     = 0.6 * temperature + np.random.normal(200, 40, n)   # correlated with temp
traffic     = np.clip(traffic, 50, 600)
deflection  = 0.04 * traffic + 0.15 * temperature + np.random.normal(0, 4, n)
print('Bridge monitoring dataset: 300 days of traffic, temperature, and deflection readings')
print(f'  Traffic range:     {traffic.min():.0f} – {traffic.max():.0f} vehicles/hr')
print(f'  Temperature range: {temperature.min():.0f} – {temperature.max():.0f} °F')
print(f'  Deflection range:  {deflection.min():.1f} – {deflection.max():.1f} mm')


## 10.1  Observational Studies vs. Controlled Experiments

A **controlled experiment** randomly assigns subjects to treatment groups so researchers can isolate cause and effect.

An **observational study** records what happens without intervening. The researcher cannot assign loads, temperatures, or traffic volumes — they can only watch.

**Structural engineering is almost entirely observational.** Engineers cannot:
- Randomly assign earthquakes to test bridges
- Deliberately collapse a building to see how much load it took
- Repeat a hurricane under controlled conditions

Instead, structural engineers instrument real structures with **strain gauges, displacement sensors, and accelerometers**, then observe what happens over time. This produces rich data — but it also creates the central challenge of observational studies: **confounding variables**.

A **confounding variable** is one that is correlated with both the variable you are studying (the explanatory variable) and the outcome — making it look like the explanatory variable is the cause when it may not be.


## 🔬 Interactive Experiment 1: Traffic, Temperature, and Bridge Deflection

The dataset above comes from a simulated bridge monitoring system. Every day for 300 days, three values were recorded:

- **Traffic volume** (vehicles per hour) — the load applied to the bridge
- **Air temperature** (°F) — a background condition
- **Midspan deflection** (mm) — how much the bridge bends under load

An engineer looks at the data and notices that deflection rises strongly on busy traffic days. They conclude: *traffic is causing the deflection.* But temperature is also higher in summer — and the bridge expands thermally, which changes its measured deflection independently of traffic.

Use the dropdown to explore the relationships in the data.


In [ ]:
def _obs_plot(view):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if view == 'Deflection vs. Traffic (raw)':
        r = np.corrcoef(traffic, deflection)[0,1]
        axes[0].scatter(traffic, deflection, c='steelblue', alpha=0.5, s=20)
        axes[0].set_xlabel('Traffic Volume (vehicles/hr)', fontsize=12)
        axes[0].set_ylabel('Midspan Deflection (mm)', fontsize=12)
        axes[0].set_title(f'Deflection vs. Traffic  (r = {r:.2f})', fontsize=13)
        axes[0].annotate('Strong correlation!\nBut is traffic the real cause?',
            xy=(0.05,0.82), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
        r2 = np.corrcoef(temperature, deflection)[0,1]
        axes[1].scatter(temperature, deflection, c='coral', alpha=0.5, s=20)
        axes[1].set_xlabel('Temperature (°F)', fontsize=12)
        axes[1].set_ylabel('Midspan Deflection (mm)', fontsize=12)
        axes[1].set_title(f'Deflection vs. Temperature  (r = {r2:.2f})', fontsize=13)

    else:  # control for temperature
        from numpy.polynomial import polynomial as P
        # Residuals after removing temperature effect
        coef = np.polyfit(temperature, deflection, 1)
        defl_resid = deflection - np.polyval(coef, temperature)
        coef2 = np.polyfit(temperature, traffic, 1)
        traffic_resid = traffic - np.polyval(coef2, temperature)
        r = np.corrcoef(traffic_resid, defl_resid)[0,1]
        axes[0].scatter(traffic_resid, defl_resid, c='steelblue', alpha=0.5, s=20)
        axes[0].set_xlabel('Traffic (temperature removed)', fontsize=12)
        axes[0].set_ylabel('Deflection (temperature removed)', fontsize=12)
        axes[0].set_title(f'After Controlling for Temperature  (r = {r:.2f})', fontsize=13)
        axes[0].annotate('Correlation drops — temperature\nwas a confounding variable!',
            xy=(0.05,0.82), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.9))
        axes[1].scatter(temperature, traffic, c='coral', alpha=0.5, s=20)
        axes[1].set_xlabel('Temperature (°F)', fontsize=12)
        axes[1].set_ylabel('Traffic Volume (vehicles/hr)', fontsize=12)
        axes[1].set_title('Why Temperature Confounds: Traffic vs. Temperature', fontsize=13)
        axes[1].annotate('Traffic IS correlated\nwith temperature — more\ncommuters in mild weather',
            xy=(0.05,0.72), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.9))

    plt.tight_layout()
    plt.show()

w_view = widgets.Dropdown(
    options=['Deflection vs. Traffic (raw)', 'Control for Temperature (residuals)'],
    description='View:', style={'description_width':'initial'},
    layout=widgets.Layout(width='400px'))
out1 = widgets.interactive_output(_obs_plot, {'view': w_view})
display(widgets.VBox([w_view, out1]))


### 💬 Stop and Think

1. Before controlling for temperature, what would an engineer conclude about traffic and deflection?
2. After controlling, how does the story change? What is the actual cause of high deflections on hot summer days?
3. Can you think of another pair of variables in structural engineering that might be correlated but not causally linked?


<div style='background:#d4edda;padding:15px;border-left:5px solid #28a745;margin:10px 0'>

### ✅ Answer Key — Stop and Think

**1. Before controlling for temperature:** The raw correlation between traffic and deflection is approximately *r* ≈ 0.85 — very strong. An engineer would reasonably conclude that heavy traffic causes high deflection. This is partially correct (traffic *does* apply load), but the correlation is inflated because temperature is hidden in the background.

**2. After controlling for temperature:** The partial correlation drops to approximately *r* ≈ 0.35–0.45. Temperature was confounding both variables: warm days bring more commuters (higher traffic) AND cause thermal expansion of the bridge deck (higher deflection). Once temperature is accounted for, the traffic-deflection relationship is weaker and more honest. The actual cause of high deflection on hot days is a *combination* of traffic load and thermal expansion — not traffic alone.

**3. Student answers vary.** Strong examples for full credit:
- Bridge age and number of reported cracks — both are driven by cumulative traffic volume and maintenance budget, not by age alone (a well-maintained 50-year-old bridge may have fewer cracks than a neglected 20-year-old one)
- Column height and foundation cost — both correlate with building size (footprint), not with each other directly
- Ice cream sales and drowning rates — both driven by heat/summer season (classic lurking variable example)
- Concrete strength and curing temperature — both are affected by mix design and climate

**Common student error:** Confusing the *direction* of confounding (thinking the confound makes the correlation look weaker, not stronger). Emphasize: confounders typically inflate correlations by creating a backdoor path between the two variables.
</div>


---
## ⚠️  Real-World Case: The I-35W Mississippi River Bridge Collapse (2007)

On August 1, 2007, the I-35W bridge in Minneapolis collapsed during the evening rush hour, killing 13 people. Over 100 were injured. At the time of collapse, approximately 570,000 pounds of construction equipment and materials were staged on the bridge deck.

**The investigation was a pure observational study.**

Engineers could not re-run the collapse. They could not randomly assign loads and see which one broke the bridge. Instead, they:
- Recovered fractured steel pieces from the river
- Reviewed 40 years of bridge inspection reports
- Analyzed photographs, sensor records, and witness accounts
- Ran computational models calibrated to observed damage patterns

The National Transportation Safety Board's conclusion: undersized **gusset plates** at the U10 joints had been carrying loads beyond their capacity since the bridge was built in 1967. Decades of inspection reports had noted corrosion and cracking — but these were interpreted as surface defects, not signs of structural overload. The confounding variable was the staging of construction equipment directly over the weakest joints.

> *Correlation in the inspection data — cracking in high-stress zones — was present for years. The causal story only became clear after the structure failed.*

This is the defining challenge of observational structural studies: you may have all the data you need to prevent a failure, but distinguishing correlation from causation requires careful analysis — not just pattern recognition.


<div style='background:#fff3cd;padding:15px;border-left:5px solid #ffc107;margin:10px 0'>

### 🍎 Teaching Note — I-35W Case Study

The gusset plate failure is a textbook confounding case: inspection reports noted cracking in high-stress zones (the outcome), but the explanatory variable (gusset plate overload) was misidentified. Engineers attributed the cracks to surface corrosion from weather — not structural overstress. This is the correlation-vs-causation problem applied to real engineering judgment.

**Key point for students:** The failure was not due to negligence or incompetence. The inspectors correctly observed the data (cracks exist). The error was in the *causal inference* — assuming weathering caused the cracks rather than asking whether stress could also explain them. This is why formal confounding analysis matters even for experienced engineers.

**Physics connection:** The gusset plates were calculated to handle a certain stress (σ = F/A). Because the plates were undersized *from original design*, the actual stress had been above the design allowable for decades. The added construction equipment (570,000 lbs) on the day of collapse was the proximate cause, but the design error was the root cause.
</div>


## 🔬 Interactive Experiment 2: When Does Correlation Mean Causation?

Not every strong correlation in structural data points to a real physical cause. The simulation below generates pairs of variables. Some pairs have a genuine cause-and-effect relationship. Others are correlated only because they share a hidden confounding variable — or by pure coincidence.

For each scenario, decide: **is this causation or confounding?**


In [ ]:
scenarios = {
    'Beam load vs. midspan deflection': {
        'x_label': 'Applied Load (kips)',
        'y_label': 'Midspan Deflection (mm)',
        'causal': True,
        'note': 'CAUSAL: Load directly causes deflection (PL³/48EI). This is physics.',
        'color': 'steelblue',
    },
    'Bridge age vs. number of cracks': {
        'x_label': 'Bridge Age (years)',
        'y_label': 'Number of Reported Cracks',
        'causal': False,
        'note': 'CONFOUNDED: Both are driven by traffic volume and maintenance budget — not age alone.',
        'color': 'coral',
    },
    'Column height vs. buckling load': {
        'x_label': 'Column Height (ft)',
        'y_label': 'Critical Buckling Load (kips)',
        'causal': True,
        'note': 'CAUSAL: Pcr = π²EI/L² — height directly determines buckling capacity. This is Euler\'s formula.',
        'color': 'steelblue',
    },
    'Number of floors vs. foundation cost': {
        'x_label': 'Number of Floors',
        'y_label': 'Foundation Cost ($M)',
        'causal': False,
        'note': 'CONFOUNDED: Both correlate with building footprint size — a taller building is also usually larger.',
        'color': 'coral',
    },
}

def _scenario_plot(scenario, reveal):
    s = scenarios[scenario]
    np.random.seed(42)
    x = np.linspace(10, 100, 80) + np.random.normal(0, 3, 80)
    if s['causal']:
        y = 0.8 * x + np.random.normal(0, 5, 80)
    else:
        confound = np.random.normal(50, 15, 80)
        x = 0.7 * confound + np.random.normal(0, 8, 80)
        y = 0.9 * confound + np.random.normal(0, 10, 80)
    r = np.corrcoef(x, y)[0, 1]
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(x, y, c=s['color'], alpha=0.6, s=35)
    ax.set_xlabel(s['x_label'], fontsize=12)
    ax.set_ylabel(s['y_label'], fontsize=12)
    ax.set_title(f"{scenario}  (r = {r:.2f})", fontsize=13)
    if reveal:
        ax.annotate(s['note'], xy=(0.03, 0.88), xycoords='axes fraction', fontsize=10,
            bbox=dict(boxstyle='round',
                facecolor='lightgreen' if s['causal'] else 'mistyrose', alpha=0.9))
    plt.tight_layout()
    plt.show()

w_scen = widgets.Dropdown(options=list(scenarios.keys()), description='Scenario:',
    style={'description_width':'initial'}, layout=widgets.Layout(width='450px'))
w_rev  = widgets.Checkbox(value=False, description='Reveal answer')
out2   = widgets.interactive_output(_scenario_plot, {'scenario': w_scen, 'reveal': w_rev})
display(widgets.VBox([w_scen, w_rev, out2]))


---
## 📋  Chapter 10 Review

| Term | Meaning |
|------|--------|
| **Observational study** | Records data without controlling the explanatory variable |
| **Controlled experiment** | Randomly assigns subjects to treatment/control groups |
| **Confounding variable** | Correlated with both the explanatory variable and the outcome |
| **Lurking variable** | A confounding variable not measured in the study |
| **Correlation** | Two variables tend to change together |
| **Causation** | One variable directly produces change in another |

**The Big Idea:** Most structural knowledge comes from observational studies — instrumented bridges, post-collapse investigations, long-term monitoring programs. Strong correlations in that data are valuable clues, but they are never proof of causation on their own. Identifying confounding variables is what separates engineering insight from pattern-matching.


<div style='background:#d1ecf1;padding:15px;border-left:5px solid #17a2b8;margin:10px 0'>

### 💬 Discussion Prompts

**1. Medical analogy (5 min):** In the 1950s, observational studies found that coffee drinkers had higher rates of lung cancer. Coffee appeared to *cause* cancer. What was the confounding variable? *(Answer: smoking — coffee drinkers at that time were more likely to smoke. Coffee and lung cancer were both correlated with smoking, not with each other.)* How does this parallel the bridge sensor scenario?

**2. Inspection failure (pair discussion):** The I-35W bridge was inspected regularly for 40 years. Why didn't those inspections catch the gusset plate problem? What would a better inspection protocol have looked like? *(Guide toward: inspections were visual; gusset plates were covered by other structural elements; inspectors lacked a model predicting which zones were overstressed.)*

**3. Sensor design (small group):** Engineers monitor real bridges with continuous sensor networks (structural health monitoring). If you were designing a monitoring system for a bridge in a climate with large temperature swings, what variables would you measure to avoid the confounding problem shown in Widget 1? *(Students should identify: temperature, traffic count AND weight, humidity, solar radiation — anything that independently affects deflection.)*

**4. Extension (homework):** Research the difference between observational studies and randomized controlled trials in medicine. Find one example where an observational study suggested a causal relationship that an RCT later reversed. *(Classic examples: hormone replacement therapy and heart disease; beta-carotene and cancer prevention.)* Write one paragraph explaining what the confounding variable was.
</div>
